# Integração das bases por bairro

**Objetivo:** juntar numa tabela só, com **uma linha por bairro**, as informações que cada pessoa da equipe coletou,
e ligar essa tabela à simulação do INCT (Tela 1).

**Como:** toda junção usa a coluna `chave`, o nome do bairro sem acento, sem espaço e em maiúsculas
(ex.: `ABERTADOSMORROS`). Ela vem do arquivo de-para de bairros feito pelo Gabriel e pela Luiza.

| Base | Responsável | Como se liga ao bairro |
|---|---|---|
| Simulação do INCT (limpa) | Gabriel | pela `chave` |
| Censo 2022 e CNEFE | Luiza | pela `chave` |
| Escolas e unidades de saúde (OpenStreetMap) | Luiggi | pela **localização no mapa** |
| Estações de Tratamento de Água (ETAs) | Luiggi | pelo **nome do bairro** nas listas do Dmae |

As bases dos colegas estão copiadas em `dados/externos/` (a origem de cada arquivo está no `README.md` dessa pasta).

In [1]:
import json
import unicodedata

import geopandas as gpd
import pandas as pd

pd.set_option("display.width", 160)

## 1. Ler as bases

In [2]:
de_para = pd.read_csv("../dados/externos/de_para_bairros.csv")
censo   = pd.read_csv("../dados/externos/censo_bairros_poa.csv")
cnefe   = pd.read_csv("../dados/externos/cnefe_resumo_bairro.csv")
inct    = pd.read_csv("../dados/externos/inct_qualidade.csv")
niveis  = pd.read_csv("../dados/externos/inct_niveis_bairro.csv")
pontos  = pd.read_csv("../dados/processados/osm_pontos_interesse_limpo.csv")
malha   = gpd.read_file("../dados/externos/bairros_poa_2016.geojson")

print("bairros no de-para:", len(de_para))
print("polígonos na malha:", len(malha), "| nomes distintos:", malha.nome.nunique())
print("pontos do OSM:", len(pontos))

bairros no de-para: 94
polígonos na malha: 128 | nomes distintos: 94
pontos do OSM: 581


A malha tem mais polígonos do que bairros porque alguns bairros têm partes separadas:
o Arquipélago tem 34 ilhas e Boa Vista do Sul tem 2 partes.

## 2. Descobrir o bairro de cada escola e unidade de saúde

Os pontos do OSM só têm latitude e longitude. Para saber o bairro, usamos uma **junção espacial** (`sjoin`):
para cada ponto, o computador verifica dentro de qual polígono de bairro ele está.

Antes disso:
1. `dissolve` junta os polígonos de mesmo nome, para ficar 1 polígono por bairro;
2. os pontos (no sistema de coordenadas do GPS, **WGS 84**) são convertidos para o sistema da malha da
   Prefeitura (**SIRGAS 2000**), porque só dá para comparar posições na mesma "régua".

In [3]:
malha = malha.dissolve(by="nome", as_index=False)
malha = malha.merge(de_para[["nome_prefeitura", "chave_tres_bases"]],
                    left_on="nome", right_on="nome_prefeitura")
malha = malha.rename(columns={"chave_tres_bases": "chave"})

pontos = gpd.GeoDataFrame(pontos,
                          geometry=gpd.points_from_xy(pontos.lon, pontos.lat),
                          crs="EPSG:4326").to_crs(malha.crs)
pontos = gpd.sjoin(pontos, malha[["chave", "geometry"]], predicate="within")

print("bairros na malha:", len(malha), "| sistema:", malha.crs)
print("pontos com bairro:", len(pontos))

bairros na malha: 94 | sistema: EPSG:4674
pontos com bairro: 581


**Conferência:** os 585 pontos encontraram um bairro. Se algum ponto ficasse fora de todos os polígonos,
ele sumiria aqui, e o número seria menor que 585.

## 3. Contar os pontos por bairro

In [4]:
contagem = pontos.groupby(["chave", "categoria"]).size().unstack(fill_value=0)
contagem.head()

categoria,escolas,saude
chave,,
ABERTADOSMORROS,3,0
AGRONOMIA,4,0
ANCHIETA,2,0
ARQUIPELAGO,7,2
AUXILIADORA,2,2


## 4. Ligar as ETAs aos bairros

As listas do Dmae misturam bairros oficiais com vilas e loteamentos, com grafias diferentes.
Para comparar, transformamos cada nome na mesma `chave` usada no de-para (sem acento, maiúsculas, só letras).

Oito nomes não são bairros oficiais, mas parecem variações de bairros oficiais que, sem eles, ficariam sem ETA.
Essa correspondência foi decidida pela equipe e está marcada como **a confirmar** com o Dmae.
Os nomes que sobram (vilas e loteamentos, como Vila Safira) ficam sem bairro.

In [5]:
def para_chave(nome):
    sem_acento = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    return "".join(letra for letra in sem_acento.upper() if letra.isalpha())

# nome na lista do Dmae -> chave do bairro oficial (a confirmar com o Dmae)
DEPARA_MANUAL = {
    "Centro": "CENTROHISTORICO",
    "Aparício Borges": "CELAPARICIOBORGES",
    "Assunção": "VILAASSUNCAO",
    "Vila Farrapos": "FARRAPOS",
    "São José": "VILASAOJOSE",
    "Alto Teresópolis": "TERESOPOLIS",   # confiança baixa: prefixo diferente
    "São Pedro": "JARDIMSAOPEDRO",       # confiança baixa: prefixo diferente
    "Vila Floresta": "JARDIMFLORESTA",   # confiança baixa: prefixo diferente
}

chaves_oficiais = set(de_para.chave_tres_bases)
with open("../dados/originais/etas_poa_enchente_2024.json", encoding="utf-8") as f:
    etas = json.load(f)["etas"]

linhas = []
for eta in etas:
    parou = eta["enchente_2024"]["parada_por_agua"]   # True, False ou None (fontes divergem)
    for nome in eta["bairros_conforme_fonte"]["lista"]:
        if para_chave(nome) in chaves_oficiais:
            chave, metodo = para_chave(nome), "nome exato"
        elif nome in DEPARA_MANUAL:
            chave, metodo = DEPARA_MANUAL[nome], "de-para manual (a confirmar)"
        else:
            chave, metodo = None, "sem bairro oficial"
        linhas.append({"eta": eta["nome"], "nome_na_fonte": nome, "chave": chave,
                       "metodo": metodo, "parou_por_agua_2024": parou})

eta_bairro = pd.DataFrame(linhas)
eta_bairro.metodo.value_counts()

metodo
nome exato                      92
sem bairro oficial              19
de-para manual (a confirmar)     8
Name: count, dtype: int64

In [6]:
print("nomes sem bairro oficial:", sorted(eta_bairro[eta_bairro.chave.isna()].nome_na_fonte.unique()))

nomes sem bairro oficial: ['Alto Petrópolis', 'Chácara da Fumaça', 'Intercap', 'Jardim Ipu', 'Jardim Planalto', 'Jardim Verde Ipanema', 'Loteamento Timbaúva', 'Parque Bahamas', 'Parque Charruas', 'Praça Moema', 'Protásio Alves', 'São Jorge', 'Vila Alto Erechim', 'Vila Campo da Tuca', 'Vila Safira', 'Vila Topázio', 'Vila dos Comerciários', 'Vila dos Sargentos']


Cada bairro pode ser atendido por mais de uma ETA. Para resumir por bairro:
- **sim**: alguma ETA do bairro parou por causa da água em 2024;
- **não confirmado**: a ETA é a São João, em que as fontes divergem sobre a causa;
- **não**: as ETAs do bairro não pararam por água;
- **sem ETA na lista**: o bairro não aparece em nenhuma lista do Dmae.

Quando há duas ETAs, "sim" **não** significa que o bairro inteiro ficou sem água.

In [7]:
def resumo_parada(valores):
    if (valores == True).any():
        return "sim"
    if valores.isna().any():
        return "não confirmado"
    return "não"

etas_por_bairro = (eta_bairro.dropna(subset=["chave"])
                   .groupby("chave")
                   .agg(etas=("eta", lambda s: "; ".join(sorted(set(s)))),
                        eta_parou_por_agua_2024=("parou_por_agua_2024", resumo_parada)))
etas_por_bairro.head()

,etas,eta_parou_por_agua_2024
chave,,
ABERTADOSMORROS,ETA Belém Novo,não
AGRONOMIA,ETA Belém Novo; ETA Menino Deus,sim
ANCHIETA,ETA São João,não confirmado
ARQUIPELAGO,ETA Ilhas,sim
AUXILIADORA,ETA Moinhos de Vento; ETA São João,sim


## 5. Juntar tudo numa tabela por bairro

`merge` é o "PROCV" do pandas. O argumento `validate="1:1"` faz o pandas **dar erro** se algum bairro aparecer
repetido em uma das tabelas, o que evitaria multiplicar linhas sem perceber.

In [8]:
tabela = (censo[["chave", "nome_inct", "pessoas", "domicilios"]]
          .merge(cnefe[["nome_inct", "pct_residencial"]], on="nome_inct", validate="1:1")
          .merge(inct[["chave", "total_buildings", "nivel_primeiro_alagamento", "pct_pop_est_5m"]],
                 on="chave", validate="1:1")
          .merge(contagem, left_on="chave", right_index=True, how="left", validate="1:1")
          .merge(etas_por_bairro, left_on="chave", right_index=True, how="left", validate="1:1"))

tabela = tabela.rename(columns={"nome_inct": "bairro", "total_buildings": "inct_total_predios",
                                "nivel_primeiro_alagamento": "inct_nivel_primeiro_alagamento_m",
                                "pct_pop_est_5m": "inct_pct_pop_afetada_5m"})
for col in ["escolas", "saude"]:
    tabela[col] = tabela[col].fillna(0).astype(int)
tabela["eta_parou_por_agua_2024"] = tabela["eta_parou_por_agua_2024"].fillna("sem ETA na lista")

print(tabela.shape)
tabela.head()

(94, 12)


,chave,bairro,pessoas,domicilios,pct_residencial,inct_total_predios,inct_nivel_primeiro_alagamento_m,inct_pct_pop_afetada_5m,escolas,saude,etas,eta_parou_por_agua_2024
0,MEDIANEIRA,Medianeira,8749,4605,90.06,4047,NaN,0.000,5,0,ETA Menino Deus,sim
1,PRAIADEBELAS,Praia de Belas,1522,1241,85.87,290,3.0,100.000,0,0,ETA Menino Deus; ETA Moinhos de Vento,sim
2,CIDADEBAIXA,Cidade Baixa,13014,10209,90.33,2339,3.3,44.936,5,1,ETA Menino Deus; ETA Moinhos de Vento,sim
3,MENINODEUS,Menino Deus,27961,17613,92.82,5522,3.3,53.657,8,2,ETA Menino Deus; ETA Moinhos de Vento,sim
4,FARROUPILHA,Farroupilha,774,519,82.25,227,7.2,0.000,7,6,ETA Moinhos de Vento,sim


`inct_nivel_primeiro_alagamento_m` fica vazio nos bairros que não alagam em nenhum nível da simulação (até 10 m).

## 6. Ligar à Tela 1: equipamentos em bairros atingidos, por nível do Guaíba

Para cada nível da simulação (3,0 a 10,0 m), combinamos a contagem de equipamentos de cada bairro com a
porcentagem de prédios afetados (`pct_predios`) naquele nível. São duas medidas:

- **`em_bairros_atingidos`**: soma os equipamentos dos bairros com algum prédio afetado. É um **teto**:
  um bairro com 1% dos prédios afetados conta todas as suas escolas.
- **`estimativa_proporcional`**: soma (equipamentos do bairro × % de prédios afetados). Supõe que os
  equipamentos estão distribuídos como os prédios.

Exemplo: um bairro com 10 escolas e 20% dos prédios afetados conta 10 na primeira medida e 2 na segunda.
Nenhuma das duas diz que uma escola específica alagou: o INCT informa quantos prédios são atingidos, não quais.

In [9]:
equip = contagem.stack().rename("n").reset_index()            # chave, categoria, n
por_nivel = niveis[["nivel_m", "chave", "pct_predios"]].merge(equip, on="chave")

por_nivel["em_bairros_atingidos"] = por_nivel.n.where(por_nivel.pct_predios > 0, 0)
por_nivel["estimativa_proporcional"] = por_nivel.n * por_nivel.pct_predios / 100

equip_nivel = (por_nivel.groupby(["nivel_m", "categoria"])[["em_bairros_atingidos", "estimativa_proporcional"]]
               .sum().round(1).reset_index())
equip_nivel[equip_nivel.nivel_m.isin([3.0, 5.3, 10.0])]

,nivel_m,categoria,em_bairros_atingidos,estimativa_proporcional
0,3.0,escolas,103,13.2
1,3.0,saude,28,2.9
46,5.3,escolas,158,57.1
47,5.3,saude,38,11.1
140,10.0,escolas,246,110.7
141,10.0,saude,89,33.5


## 7. Conferências e gravação

In [10]:
print("bairros na tabela:", len(tabela), "| chaves únicas:", tabela.chave.is_unique)
print("escolas:", tabela.escolas.sum(), "| saúde:", tabela.saude.sum(), "| total:", len(pontos))
print("população total (Censo 2022):", tabela.pessoas.sum())
print("bairros por situação das ETAs:", tabela.eta_parou_por_agua_2024.value_counts().to_dict())

bairros na tabela: 94 | chaves únicas: True
escolas: 433 | saúde: 148 | total: 581
população total (Censo 2022): 1332845
bairros por situação das ETAs: {'sim': 48, 'não confirmado': 25, 'não': 11, 'sem ETA na lista': 10}


In [11]:
from pathlib import Path

saida = Path("../dados/processados/integracao")
saida.mkdir(parents=True, exist_ok=True)
tabela.to_csv(saida / "bairros_integrado.csv", index=False)
eta_bairro.to_csv(saida / "eta_bairro.csv", index=False)
equip_nivel.to_csv(saida / "equipamentos_por_nivel.csv", index=False)
print(sorted(p.name for p in saida.iterdir()))

['bairros_integrado.csv', 'equipamentos_por_nivel.csv', 'eta_bairro.csv']


## Limitações

- **Simulação do INCT:** os números são estimativas de uma simulação, não previsões nem contagens oficiais.
- **OpenStreetMap:** é mapeado por voluntários; as contagens são o que está mapeado, não o total real de escolas e
  unidades de saúde.
- **ETAs:** as listas do Dmae não cobrem todos os bairros e incluem vilas que não são bairros oficiais.
  As 8 correspondências manuais precisam ser confirmadas com o Dmae.
- **Pendente:** comparar a simulação com a mancha de inundação observada em maio de 2024 (base do Lucas).

*Notebook feito com auxílio do Claude Code, conforme exigido no enunciado.*